In [ ]:
import sys
import pyomo
import pandas as pd

from pyomo.environ import *
solver = SolverFactory('cbc')


UC

Model, Parameter

In [ ]:
from pyomo.environ import*

# =====================
# Model
# =====================
model = ConcreteModel()

# =====================
# Sets
# =====================
model.G = Set(initialize=['G1','G2','G3'])
model.T = Set(initialize=[1,2,3,4,5,6,7,8])

# =====================
# Parameters
# =====================
Pmin = {'G1':20, 'G2':30, 'G3':50}
Pmax = {'G1':150, 'G2':120, 'G3':100}
Cost = {'G1':15, 'G2':15, 'G3':12.35}
StartupCost = {'G1':500, 'G2':400, 'G3':300}
ShutdownCost = {'G1':50, 'G2':40, 'G3':30}
RampUp = {'G1':50, 'G2':50, 'G3':50}
RampDown = {'G1':50, 'G2':100, 'G3':100}
MinUp = {'G1':2, 'G2':2, 'G3':2}
MinDown = {'G1':2, 'G2':2, 'G3':2}

Demand = {1:150, 2:250, 3:220, 4:200, 5:300, 6:160, 7:300, 8:50}

ReserveReq = {t: 0.10 * Demand[t] for t in model.T}

model.Pmin = Param(model.G, initialize=Pmin)
model.Pmax = Param(model.G, initialize=Pmax)
model.Cost = Param(model.G, initialize=Cost)
model.Demand = Param(model.T, initialize=Demand)
model.StartupCost = Param(model.G, initialize=StartupCost)
model.ShutdownCost = Param(model.G, initialize=ShutdownCost)
model.RampUp = Param(model.G, initialize=RampUp)
model.RampDown = Param(model.G, initialize=RampDown)
model.MinUp = Param(model.G, initialize=MinUp)
model.MinDown = Param(model.G, initialize=MinDown)
model.ReserveReq = Param(model.T, initialize=ReserveReq)

Variables

In [ ]:
# =====================
# Variables
# =====================
model.P = Var(model.G, model.T, domain=NonNegativeReals)
model.u = Var(model.G, model.T, domain=Binary)
model.y = Var(model.G, model.T, domain=Binary)              #startup
model.z = Var(model.G, model.T, within=Binary)              #shutdown
model.R = Var(model.G, model.T, within=NonNegativeReals)    #spinning reserve

Objective

In [ ]:
# =====================
# Objective
# =====================
def obj_rule(m):
    return sum(m.Cost[g] * m.P[g,t] 
               + m.StartupCost[g] * m.y[g,t]
               + m.ShutdownCost[g] * m.z[g,t]
               for g in m.G for t in m.T)

model.Obj = Objective(rule=obj_rule, sense=minimize)

Constraints

In [ ]:
# =====================
# Constraints
# =====================

# Power balance
def balance_rule(m, t):
    return sum(m.P[g,t] for g in m.G) == m.Demand[t]

model.Balance = Constraint(model.T, rule=balance_rule)

# Capacity upper bound
def max_rule(m, g, t):
    return m.P[g,t] <= m.Pmax[g] * m.u[g,t]

model.MaxCap = Constraint(model.G, model.T, rule=max_rule)

# Capacity lower bound
def min_rule(m, g, t):
    return m.P[g,t] >= m.Pmin[g] * m.u[g,t]

model.MinCap = Constraint(model.G, model.T, rule=min_rule)

# Startup Logic (Revisi)
def transition_rule(m, g, t):
    if t == 1:
        return m.u[g,t] - 0 == m.y[g,t] - m.z[g,t]
    else:
        return m.u[g,t] - m.u[g,t-1] == m.y[g,t] - m.z[g,t]

model.Transition = Constraint(model.G, model.T, rule=transition_rule)

# Ramp Up
def ramp_up_rule(m, g, t):
    if t == 1:
        return Constraint.Skip
    return m.P[g,t] - m.P[g,t-1] <= m.RampUp[g]

model.RampUpConstraint = Constraint(model.G, model.T, rule=ramp_up_rule)

# Ramp Down
def ramp_down_rule(m, g, t):
    if t == 1:
        return Constraint.Skip
    return m.P[g,t-1] - m.P[g,t] <= m.RampDown[g]

model.RampDownConstraint = Constraint(model.G, model.T, rule=ramp_down_rule)

# Minimum Up Time
def min_up_rule(m, g, t):
    if t + m.MinUp[g] - 1 > max(m.T):
        return Constraint.Skip
    return sum(m.u[g,k] for k in range(t, t + m.MinUp[g])) \
           >= m.MinUp[g] * m.y[g,t]

model.MinUpConstraint = Constraint(model.G, model.T, rule=min_up_rule)

# Minimum Down Time
def min_down_rule(m, g, t):
    if t + m.MinDown[g] - 1 > max(m.T):
        return Constraint.Skip
    return sum(1 - m.u[g,k] for k in range(t, t + m.MinDown[g])) \
           >= m.MinDown[g] * m.z[g,t]

model.MinDownConstraint = Constraint(model.G, model.T, rule=min_down_rule)

# Reverse Generator
def reserve_cap_rule(m, g, t):
    return m.P[g,t] + m.R[g,t] <= m.Pmax[g] * m.u[g,t]

model.ReserveCap = Constraint(model.G, model.T, rule=reserve_cap_rule)

# Reverse Sistem
def system_reserve_rule(m, t):
    return sum(m.R[g,t] for g in m.G) >= m.ReserveReq[t]

model.SystemReserve = Constraint(model.T, rule=system_reserve_rule)

Solve

In [ ]:
# =====================
# Solve
# =====================
solver = SolverFactory('cbc')
results = solver.solve(model)

Results

In [ ]:
# =====================
# Format
# =====================
def fmt(x):
    if abs(x - round(x)) < 1e-6:
        return int(round(x))
    else:
        return round(x, 2)
    
# =====================
# Print Results
# =====================
#print("Total System Cost =", fmt(value(model.Obj)))

from pyomo.opt import TerminationCondition
results = solver.solve(model)

if results.solver.termination_condition != TerminationCondition.optimal:
    print("Model infeasible or not optimal.")
else:
    print("Optimal solution found.")

rows = []

for t in model.T:
    g1_p = value(model.P['G1', t])
    g2_p = value(model.P['G2', t])
    g3_p = value(model.P['G3', t])
     # Fuel cost
    g1_fuel = g1_p * value(model.Cost['G1'])
    g2_fuel = g2_p * value(model.Cost['G2'])
    g3_fuel = g3_p * value(model.Cost['G3'])
    
    fuel_cost = g1_fuel + g2_fuel + g3_fuel
    
    # Startup cost
    g1_start = value(model.y['G1', t]) * value(model.StartupCost['G1'])
    g2_start = value(model.y['G2', t]) * value(model.StartupCost['G2'])
    g3_start = value(model.y['G3', t]) * value(model.StartupCost['G3'])
    
    startup_cost = g1_start + g2_start + g3_start
    
    total_cost = fuel_cost + startup_cost

    # Reserve
    r1 = value(model.R['G1', t])
    r2 = value(model.R['G2', t])
    r3 = value(model.R['G3', t])

    total_reserve = r1 + r2 + r3
    reserve_req = value(model.ReserveReq[t])
    margin = total_reserve - reserve_req
    
    
    rows.append([
        t,
        fmt(value(model.Demand[t])),
        fmt(g1_p),
        fmt(g2_p),
        fmt(g3_p),

        fmt(r1),
        fmt(r2),
        fmt(r3),
        fmt(total_reserve),
        fmt(reserve_req),
        fmt(margin),
        
        fmt(fuel_cost),
        fmt(startup_cost),
        fmt(total_cost)
    ])

df_hourly = pd.DataFrame(
    rows,
    columns=[
        "Hour",
        "Demand",
        "G1_P", "G2_P", "G3_P",

        "R_G1", "R_G2", "R_G3",
        "Total_Reserve",
        "Reserve_Req",
        "Reserve_Margin",

        "Fuel Cost",
        "Startup Cost",
        "Total Cost"
    ]
)

print("\n=== Hourly UC Result ===")
print(df_hourly.to_string(index=False))

# ==========================
# TABEL RAMPING
# ==========================

ramp_rows = []

for t in model.T:
    if t == 1:
        ramp_rows.append([t, "-", "-", "-"])
    else:
        dp1 = value(model.P['G1', t]) - value(model.P['G1', t-1])
        dp2 = value(model.P['G2', t]) - value(model.P['G2', t-1])
        dp3 = value(model.P['G3', t]) - value(model.P['G3', t-1])
        
        ramp_rows.append([
            t,
            fmt(dp1),
            fmt(dp2),
            fmt(dp3)
        ])

df_ramp = pd.DataFrame(
    ramp_rows,
    columns=["Hour", "ΔG1", "ΔG2", "ΔG3"]
)

print("\n=== Ramping Check (ΔP) ===")
print(df_ramp.to_string(index=False))

# ==========================
# SYSTEM SUMMARY
# ==========================

peak_demand = max(value(model.Demand[t]) for t in model.T)
total_cost_system = value(model.Obj)

print("\n=== System Summary ===")
print("Peak Demand =", fmt(peak_demand))
print("Total System Cost =", fmt(total_cost_system))

Optimal solution found.

=== Hourly UC Result ===
 Hour  Demand  G1_P  G2_P  G3_P  R_G1  R_G2  R_G3  Total_Reserve  Reserve_Req  Reserve_Margin  Fuel Cost  Startup Cost  Total Cost
    1     150    50     0   100    15     0     0             15           15               0       1985           800        2785
    2     250   100    50   100    25     0     0             25           25               0       3485           400        3885
    3     220    90    30   100     0    22     0             22           22               0       3035             0        3035
    4     200    50    50   100   100     0     0            100           20              80       2735             0        2735
    5     300   100   100   100    30     0     0             30           30               0       4235             0        4235
    6     160    50    50    60   100    70    40            210           16             194       2241             0        2241
    7     300   100   100   100  